# Confirmação dos Insights para Looker Studio

Notebook com queries individuais para validação dos 7 principais insights de clientes extraídos pela pipeline (Regra 6 a 10). Cada célula pode ser executada independentemente para conferir os valores que alimentam o dashboard no Looker Studio.



# Notebook 08_confirmação_looker

Cada célula traz uma das queries para confirmar os dados que alimentam o dashboard do Looker Studio. Validado pela última vez em 22/10/2026.

**Parte 1 — Insights das regras (views UC):**
- Regra 6: Novos clientes
- Regra 7a: Provedores desconhecidos
- Regra 7b: Provedor líder
- Regra 8: E-mails duplicados
- Regra 9a: Tempo até primeira compra
- Regra 9b: Compra antes do cadastro
- Regra 10: Atualização anterior ao cadastro

**Parte 2 — Atualizações de 22/10:**
- `vw_provedor_lider_detalhe`: gráfico de provedores (barras + percentual)
- `vw_clientes_por_ano`: cadastrados vs compraram por ano
- `vw_compradores_por_ano`: compradores por ano (visão dos pedidos)
- `vw_fontes_compradores`: auditoria de proveniência (9.998 de 10.002 compradores sem cadastro)
- Fonte em tempo real: `squad2.gold_insights_clientes` + histórico no SQL Server (jobs de 15 min)

A esteira automática (jobs `Gold KPIs` e `insights_clientes_atualizacao_15min`) atualiza tudo a cada 15 minutos — este notebook serve para conferência pontual.

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'novos_clientes';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'provedores_desconhecidos';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'provedor_lider';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'email_duplicado';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'tempo_primeira_compra';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'compra_antes_cadastro';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes WHERE insight = 'atualizacao_anterior_cadastro';

In [0]:
%sql
SELECT * FROM workspace.default.vw_insights_clientes ORDER BY regra, insight;

In [0]:
%sql
SELECT * FROM workspace.default.vw_clientes_compras


In [0]:
%sql
-- Clientes cadastrados, sem compra e com compra
CREATE OR REPLACE VIEW workspace.default.vw_clientes_compras_detalhado AS
SELECT
  COUNT(*) AS total_cadastrados,
  COUNT(*) - COUNT(DISTINCT CASE WHEN p.id_cliente IS NOT NULL THEN c.id_cliente END) AS sem_compra,
  COUNT(DISTINCT CASE WHEN p.id_cliente IS NOT NULL THEN c.id_cliente END) AS comprou
FROM workspace.default.silver_ecommerce_clientes c
LEFT JOIN workspace.default.silver_ecommerce_pedidos p
  ON c.id_cliente = p.id_cliente;

-- Visualizar
SELECT * FROM workspace.default.vw_clientes_compras_detalhado;

## Atualizações de 22/10 — validação das novas fontes
Conferência das views criadas para o dashboard e da fonte em tempo real do SQL Server (alimentada pelos jobs de 15 min).

In [0]:
%sql
-- Gráfico de provedores: dimensão provedor, métricas quantidade e percentual
SELECT * FROM workspace.default.vw_provedor_lider_detalhe ORDER BY quantidade DESC;

In [0]:
%sql
-- Cadastrados vs compraram por ano (base: Silver de clientes)
SELECT * FROM workspace.default.vw_clientes_por_ano ORDER BY ano;

In [0]:
%sql
-- Compradores por ano da primeira compra (visão dos pedidos; rotular como "Compradores (pedidos)")
SELECT * FROM workspace.default.vw_compradores_por_ano ORDER BY ano;

In [0]:
%sql
-- Auditoria de proveniência: fontes de clientes testadas contra os 10.002 compradores
SELECT * FROM workspace.default.vw_fontes_compradores ORDER BY total_clientes DESC;

In [0]:
%sql
-- Fonte em tempo real do Looker: frescor das tabelas mantidas pelos jobs de 15 min
SELECT
  (SELECT MAX(calculado_em) FROM sqlserver_catalog.squad2.gold_insights_clientes) AS insights_ultima_atualizacao,
  (SELECT COUNT(*) FROM sqlserver_catalog.squad2.gold_insights_clientes_historico) AS historico_linhas,
  (SELECT MAX(calculado_em) FROM sqlserver_catalog.squad2.gold_insights_clientes_historico) AS historico_ultimo_ciclo;

In [0]:
%sql
-- Insights atuais na fonte do dashboard (SQL Server, janelas de tempo — não comparar com as views UC)
SELECT regra, insight, titulo, mensagem, valor, severidade, calculado_em
FROM sqlserver_catalog.squad2.gold_insights_clientes
ORDER BY regra, insight;

## Fontes para colar no Looker Studio (modo "Consulta personalizada")
Use estas queries ao criar as fontes novas — na conexão Databricks que já funciona, marque **Consulta personalizada** e cole. As views UC (`vw_insights_clientes` etc.) **não precisam ser reinseridas** — já alimentam os cards cumulativos.

In [0]:
%sql
-- FONTE A — Tempo real (janelas 10/60 min): para os cards de cima
-- No Looker: colar em "Consulta personalizada"; filtrar cada card por insight = ...
SELECT regra, insight, titulo, mensagem, valor, severidade, calculado_em
FROM sqlserver_catalog.squad2.gold_insights_clientes
ORDER BY regra, insight;

In [0]:
%sql
-- FONTE B — Histórico (evolução): para a série temporal + filtro de período
-- No Looker: dimensão calculado_em, métrica valor (AVG), filtro insight = ...
SELECT calculado_em, regra, insight, titulo, valor, severidade
FROM sqlserver_catalog.squad2.gold_insights_clientes_historico
ORDER BY calculado_em;

In [0]:
%sql
-- Amostra da evolução (últimos 10 ciclos) — prévia do que a série temporal mostrará
SELECT * FROM (
  SELECT calculado_em, insight, valor
  FROM sqlserver_catalog.squad2.gold_insights_clientes_historico
  WHERE insight = 'tempo_primeira_compra'
  ORDER BY calculado_em DESC
  LIMIT 10
) t ORDER BY calculado_em;

In [0]:
%sql
-- Checagem 363 vs 366: linhas vs clientes únicos na Silver + ciclo atual do card UC
SELECT
  (SELECT COUNT(*) FROM workspace.default.silver_ecommerce_clientes) AS linhas_silver,
  (SELECT COUNT(DISTINCT id_cliente) FROM workspace.default.silver_ecommerce_clientes) AS clientes_unicos,
  (SELECT valor FROM workspace.default.gold_insights_clientes WHERE insight = 'novos_clientes' LIMIT 1) AS card_uc,
  (SELECT MAX(calculado_em) FROM workspace.default.gold_insights_clientes) AS uc_ultimo_ciclo;

In [0]:
%sql
-- FONTE C — Clientes cadastrados desde o início X compradores com/sem cadastro
-- No Looker: gráfico de colunas com as 3 métricas
SELECT
  (SELECT COUNT(*) FROM workspace.default.silver_ecommerce_clientes) AS cadastrados_desde_inicio,
  (SELECT COUNT(DISTINCT p.id_cliente)
     FROM workspace.default.silver_ecommerce_pedidos p
     WHERE EXISTS (SELECT 1 FROM workspace.default.silver_ecommerce_clientes c
                   WHERE c.id_cliente = p.id_cliente)) AS compraram_com_cadastro,
  (SELECT COUNT(DISTINCT p.id_cliente)
     FROM workspace.default.silver_ecommerce_pedidos p
     WHERE NOT EXISTS (SELECT 1 FROM workspace.default.silver_ecommerce_clientes c
                       WHERE c.id_cliente = p.id_cliente)) AS compraram_sem_cadastro;

In [0]:
%sql
-- Gráfico: visão geral de clientes desde o início (cadastrados / compraram com cadastro / sem cadastro)
CREATE OR REPLACE VIEW workspace.default.vw_clientes_visao_geral AS
SELECT
  (SELECT COUNT(*) FROM workspace.default.silver_ecommerce_clientes) AS cadastrados_desde_inicio,
  (SELECT COUNT(DISTINCT p.id_cliente) FROM workspace.default.silver_ecommerce_pedidos p
     WHERE EXISTS (SELECT 1 FROM workspace.default.silver_ecommerce_clientes c WHERE c.id_cliente = p.id_cliente)) AS compraram_com_cadastro,
  (SELECT COUNT(DISTINCT p.id_cliente) FROM workspace.default.silver_ecommerce_pedidos p
     WHERE NOT EXISTS (SELECT 1 FROM workspace.default.silver_ecommerce_clientes c WHERE c.id_cliente = p.id_cliente)) AS compraram_sem_cadastro;

-- Visualizar
SELECT * FROM workspace.default.vw_clientes_visao_geral;